In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_19.pickle

In [ ]:
%%RecordEvent
%%time
### cell 20 ###

edu_mapping = {
    'Masterâ\x80\x99s degree': "Master's degree",
    'Bachelorâ\x80\x99s degree': "Bachelor's degree"
}
# Map raw education values to standardized labels
# Include 2022 for completeness (no matches if already clean)
dfs_cols = [
    (responses_df_2017, 'FormalEducation'),
    (responses_df_2018, question_of_interest),
    (responses_df_2019, question_of_interest),
    (responses_df_2020, question_of_interest),
    (responses_df_2021, question_of_interest),
    (responses_df_2022, question_of_interest)
]
for df, col in dfs_cols:
    df['category'] = df[col].replace(edu_mapping)

# Concatenate all years into one GPU DataFrame with a 'year' column
raw_combined = pd.concat([
    df[['category']].assign(year=yr)
    for (df, _), yr in zip(dfs_cols, ['2017','2018','2019','2020','2021','2022'])
], ignore_index=True)

# Compute counts per (year, category) and total per year
grouped = raw_combined.groupby(['year', 'category']).size().reset_index(name='count')
totals = raw_combined.groupby('year').size().reset_index(name='total')

# Merge and calculate percentages
pct_df = grouped.merge(totals, on='year')
pct_df['percentage'] = (pct_df['count'] * 100 / pct_df['total']).round(1)

# Build final DataFrame and rename 'category' to the x-axis title
degree_df_combined = pct_df[['percentage','year','category']].rename(columns={'category': title_for_x_axis})

# Group all non-target degrees into 'Other'
degree_df_combined[title_for_x_axis] = degree_df_combined[title_for_x_axis].where(
    degree_df_combined[title_for_x_axis].isin(subset_of_degrees), 'Other'
)
# Sum percentages for 'Other' by year
degree_df_combined = degree_df_combined.groupby(['year', title_for_x_axis], as_index=False)['percentage'].sum()
# Sort for presentation
degree_df_combined = degree_df_combined.sort_values(['year','percentage'])

# Output CSV for charting
bar_chart_multiple_years_29(degree_df_combined, title_for_x_axis, column_of_interest, title_for_chart, title_for_y_axis)

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_20_try_7.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_20_try_7.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[20], f)


In [ ]:
opt_output = Out.get(4)